In [1]:
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import os
import pickle

# --- Configuration ---
DATA_OUTPUT_PATH = "C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/ml_experiments_anish/Phase1 Exp/output/processed_data.pkl" 
JSONL_FILE_PATH = 'C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/analysis_pipeline/data/raw/synthetic_dataset2.jsonl'

# --- Part 1: Load Data (No changes here) ---
print(f"Loading data from {JSONL_FILE_PATH}...")
source_data = []
try:
    with open(JSONL_FILE_PATH, 'r') as f:
        for line in f:
            if line.strip():
                source_data.append(json.loads(line))
    print(f"Successfully loaded {len(source_data)} records.")
except Exception as e:
    print(f"An error occurred during file loading: {e}")
    source_data = []


# --- Part 2: FINAL CORRECTED Processing and Normalization ---
if source_data:
    processed_records = []
    skipped_count = 0
    for record in tqdm(source_data, desc="Processing records"):
        memory_id = record.get('memory_id')
        embeddings_dict = record.get('embeddings')

        if not memory_id or not isinstance(embeddings_dict, dict):
            skipped_count += 1
            continue

        # CORRECTED: Access the 'vector' key inside 'image' and 'caption'
        image_emb_list = embeddings_dict.get('image', {}).get('vector')
        caption_emb_list = embeddings_dict.get('caption', {}).get('vector')

        if not isinstance(image_emb_list, list) or not isinstance(caption_emb_list, list):
            skipped_count += 1
            continue

        try:
            image_emb = np.array(image_emb_list, dtype=np.float32)
            caption_emb = np.array(caption_emb_list, dtype=np.float32)

            image_norm = np.linalg.norm(image_emb)
            caption_norm = np.linalg.norm(caption_emb)

            if image_norm > 0 and caption_norm > 0:
                image_emb_normalized = image_emb / image_norm
                caption_emb_normalized = caption_emb / caption_norm
                processed_records.append({
                    'memory_id': memory_id,
                    'image_embedding': image_emb_normalized,
                    'caption_embedding': caption_emb_normalized
                })
            else:
                skipped_count += 1
        except Exception as e:
            skipped_count += 1

    if skipped_count > 0:
        print(f"\nSkipped {skipped_count} records due to missing or invalid data.")

    # --- Part 3: Create DataFrame and Save (No changes here) ---
    if processed_records:
        df_processed = pd.DataFrame(processed_records)

        output_dir = os.path.dirname(DATA_OUTPUT_PATH)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)

        print(f"\nProcessed {len(df_processed)} records successfully.")
        print("Saving final processed data to:", DATA_OUTPUT_PATH)
        df_processed.to_pickle(DATA_OUTPUT_PATH)

        print("\nStep 1 is complete! Data has been acquired and preprocessed.")
        print("Here's a preview of the final data structure:")
        display(df_processed.head())
    else:
        print("\nNo records were processed. Please double-check the file and paths.")

Loading data from C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/analysis_pipeline/data/raw/synthetic_dataset2.jsonl...
Successfully loaded 240 records.


Processing records:   0%|          | 0/240 [00:00<?, ?it/s]


Processed 240 records successfully.
Saving final processed data to: C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/ml_experiments_anish/Phase1 Exp/output/processed_data.pkl

Step 1 is complete! Data has been acquired and preprocessed.
Here's a preview of the final data structure:


,memory_id,image_embedding,caption_embedding
0,synth_0_000_0001,"[0.06872081, 0.07260379, -0.13583875, -0.05191...","[-0.026345456, 0.21875048, 0.07395973, 0.13623..."
1,synth_0_001_0002,"[0.16515079, 0.0784886, -0.21923748, -0.032796...","[-0.029676182, 0.25977787, 0.026610577, 0.1004..."
2,synth_0_002_0003,"[0.10920579, -0.023704186, -0.1344325, -0.0577...","[-0.069036804, 0.21500823, 0.027473819, 0.2163..."
3,synth_0_003_0004,"[0.03766705, 0.10330472, -0.17655662, -0.04844...","[-0.115095295, 0.18258934, 0.0057722973, 0.130..."
4,synth_0_004_0005,"[0.11215473, 0.09267235, -0.13421886, -0.09326...","[-0.013061556, 0.22312802, -0.020890966, 0.131..."


In [5]:
import pandas as pd
import numpy as np

# Update this path to the file you just saved
INPUT_PATH = r'C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/ml_experiments_anish/Phase1 Exp/output/processed_data.pkl'

# Load the data
df = pd.read_pickle(INPUT_PATH)

print(f"Successfully loaded {len(df)} records.")
print(df.head())


Successfully loaded 240 records.
          memory_id                                    image_embedding  \
0  synth_0_000_0001  [0.06872081, 0.07260379, -0.13583875, -0.05191...   
1  synth_0_001_0002  [0.16515079, 0.0784886, -0.21923748, -0.032796...   
2  synth_0_002_0003  [0.10920579, -0.023704186, -0.1344325, -0.0577...   
3  synth_0_003_0004  [0.03766705, 0.10330472, -0.17655662, -0.04844...   
4  synth_0_004_0005  [0.11215473, 0.09267235, -0.13421886, -0.09326...   

                                   caption_embedding  
0  [-0.026345456, 0.21875048, 0.07395973, 0.13623...  
1  [-0.029676182, 0.25977787, 0.026610577, 0.1004...  
2  [-0.069036804, 0.21500823, 0.027473819, 0.2163...  
3  [-0.115095295, 0.18258934, 0.0057722973, 0.130...  
4  [-0.013061556, 0.22312802, -0.020890966, 0.131...  

Sample Embedding Keys: [ 0.06872081  0.07260379 -0.13583875 -0.05191025 -0.03496714 -0.03606398
  0.18538193 -0.17247431  0.02010969 -0.06846139 -0.00988952 -0.11506758
  0.09816457 -0.113996